# Lab 06: Nisqually vs. Northridge — NumPy on Earthquake Data

**EES 3350/5350 · Python in Earth Science · Monday, October 12, 2026**<br>
**Due:** Sunday, October 18, at 11:59pm, submitted by pushing to your assignment repo.

Two earthquakes, seven years apart, almost the same size:

|                 |**Nisqually, Washington**|**Northridge, California**|
|:--|:--|:--|
|date|February 28, 2001|January 17, 1994|
|magnitude|6.8|6.7|
|depth|about 52 km|about 18 km|
|deaths|1|57|
|estimated losses|about \$2 billion|about \$20 billion|

*Sources: Nisqually depth and magnitude from the Pacific Northwest Seismic Network ([PNSN, 2001](https://assets.pnsn.org/legacy_reports/Sum01/q01A.html)); Nisqually deaths and the Governor's early loss estimate from the Earthquake Engineering Research Institute ([EERI, 2001](https://www.eeri.org/lfe/pdf/usa_nisqually_preliminary_report.pdf)); Northridge deaths and losses from the U.S. Geological Survey ([USGS Open-File Report 96-263](https://pubs.usgs.gov/of/1996/ofr-96-0263/introduc.htm)); Northridge depth from the USGS catalog file you'll load in Part 3.*

The **slightly bigger** earthquake did about **a tenth** of the damage. Today you'll use NumPy to find out why, using everything from the last week: element-wise math, `np.linspace()`, masks, `argmax`, `np.nan`, 2D arrays, `axis`, and broadcasting.

This lab is long on purpose. Plan on using the whole lab period.

## Getting started

1. Accept the Lab 06 assignment from the link on Brightspace.
2. In an ExploreHub terminal, clone your repo and look inside:
   ```
   cd ~/assignments
   git clone git@github.com:Python-in-Earth-Science/ees-3350-5350-lab-06-USERNAME.git
   cd ees-3350-5350-lab-06-USERNAME
   ls
   ```
   You should see this notebook, `lab06.ipynb`, and two data files: `nisqually_2001.csv` and `northridge_1994.csv`.
3. Open `lab06.ipynb` in ExploreHub and rename it `lab06_LastName.ipynb` (right-click it in the file browser, then **Rename**). Do all of your work in this notebook.
4. Write your code in the code cell under each exercise. In the first few exercises, some of the code is written for you: replace each `___` with the right code. Later exercises start from an empty cell. Answer each orange **Question** in the Markdown cell right below it (double-click the cell to edit it).

Because your notebook and the data files are in the same folder, you can load the files by name, with no `../../`.

Run this cell first, every time you open the notebook:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Part 1: How much energy?

Magnitude is a **logarithmic** scale. A widely used relation between an earthquake's magnitude $M$ and the energy it releases as seismic waves, $E$ in joules, is

$$\log_{10} E = 1.5\,M + 4.8 \qquad \text{or, equivalently,} \qquad E = 10^{\,1.5M + 4.8}$$

*This is the Gutenberg–Richter energy–magnitude relation (Gutenberg & Richter, 1956). It was built for surface-wave magnitude, so using it with other kinds of magnitude is an approximation (Okal, 2019). That's fine for comparing two earthquakes of similar size, which is what we're doing.*

In Python, $10^{x}$ is `10 ** x`.

<div class="alert alert-info">

### Exercise 1: Magnitude to energy

1. Write a function `quake_energy(magnitude)`, with a docstring, that returns the energy in joules. Test it on 6.8 and 6.7.
2. How many times more energy did Nisqually release than Northridge?
3. Exploding one ton of TNT releases $4.184 \times 10^9$ J, written `4.184e9` in Python. Convert both earthquakes' energy to tons of TNT.
4. Make `mags = np.arange(3, 9)` and pass the **whole array** to `quake_energy()` in one call to make `energies`. Then compute `energies[1:] / energies[:-1]`. What does each number in the result tell you?

</div>

In [ ]:
# your answer here
# 1
def quake_energy(magnitude):
    """Seismic energy (J) released by an earthquake of the given magnitude (Gutenberg & Richter, 1956)."""
    return ___

print(quake_energy(6.8))
print(quake_energy(___))

# 2
ratio = ___ / ___
print(f"Nisqually released {ratio:.2f} times as much energy as Northridge")

# 3
tnt = 4.184e9    # joules in one ton of TNT
print(f"Nisqually:  {___:,.0f} tons of TNT")    # ,.0f adds commas to big numbers
print(f"Northridge: {___:,.0f} tons of TNT")

# 4
mags = np.arange(3, 9)
energies = ___
print(mags)
print(energies[1:] / energies[:-1])


<details>
<summary style="color:green;"><strong>Hint for #4.</strong></summary>

`energies[1:]` is every element except the first; `energies[:-1]` is every element except the last. Both have the same length, so dividing them pairs each element with its neighbor.
</details>

<div class="alert alert-warning">

**Question 1:** Nisqually released about 40% more energy than Northridge but caused about a tenth of the losses. Based on that, is magnitude alone a good predictor of damage? What else might matter?

</div>

> your answer here

---
## Part 2: How far away was it?

An earthquake starts at a point underground called the **hypocenter**. The spot on the surface directly above it is the **epicenter**. If you're standing some **epicentral distance** away along the surface, your straight-line distance to the hypocenter makes a right triangle with the depth:

```
   you ●─────── epicentral distance ───────● epicenter (surface)
        ╲                                  │
          ╲  hypocentral distance          │ depth
            ╲                              │
              ╲                            │
                ╲──────────────────────────● hypocenter
```

$$\text{hypocentral distance} = \sqrt{(\text{epicentral distance})^2 + (\text{depth})^2}$$

<div class="alert alert-info">

### Exercise 2: A distance table with broadcasting

1. Make `depths = np.array([51.8, 18.2])`, the hypocenter depths in km of Nisqually and Northridge (from the data file headers), and `epi = np.arange(0, 60, 10)`, epicentral distances from 0 to 50 km.
2. **Predict first:** what are the shapes of `depths` and `epi`? Will `np.sqrt(epi**2 + depths**2)` work? Run it and read the error.
3. Fix it with `np.newaxis` so that `hypo` is a `(2, 6)` table: one **row** per earthquake, one **column** per epicentral distance. Print its shape.
4. Print each row with a label, rounded to one decimal.
5. Someone standing **right on top of** Nisqually's epicenter was 51.8 km from the hypocenter. How far from Northridge's **epicenter** would you need to be to be 51.8 km from **its** hypocenter? (Rearrange the formula.)

</div>

In [ ]:
# your answer here
# 1
depths = np.array([51.8, 18.2])    # km: Nisqually, Northridge
epi = np.arange(___, ___, ___)     # 0, 10, 20, 30, 40, 50 km

# 2: predict the shapes first, then run
print(depths.shape, epi.shape)
# np.sqrt(epi**2 + depths**2)      # delete the # to see the error, then put it back

# 3
hypo = np.sqrt(epi**2 + depths[___]**2)
print(hypo.shape)

# 4
print("epicentral (km):", epi)
print("Nisqually  (km):", hypo[___].round(1))
print("Northridge (km):", hypo[___].round(1))

# 5


<div class="alert alert-warning">

**Question 2:** Use your table to explain, in two or three sentences, why the deeper earthquake might have shaken the surface less, even though it released more energy.

</div>

> your answer here

<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab06_LastName.ipynb
git commit -m "Finish Exercises 1-2"
git push
```

</div>

---
## Part 3: The first week of aftershocks

Big earthquakes are followed by **aftershocks**, smaller earthquakes near the same place. The two data files list every event in the U.S. Geological Survey's catalog within 50 km of each epicenter, starting the day of the mainshock. Run the cell below to take a look at the first few lines.

|**column**|**what it holds**|
|--:|:--|
|0|`hours`: time after the mainshock (negative = before it)|
|1|latitude (°N)|
|2|longitude (°E; negative = west)|
|3|depth (km)|
|4|magnitude (`nan` = not reported)|
|5|`is_earthquake`: 1 = earthquake, 0 = explosion|

*Data: U.S. Geological Survey, Advanced National Seismic System Comprehensive Catalog ([ComCat](https://earthquake.usgs.gov/data/comcat/)), accessed October 11, 2026. The exact search for each file is in its header.*

**Explosions?** Mines and quarries set off blasts that seismometers record, and the catalog keeps them so they aren't mistaken for earthquakes. We need to remove them ourselves.

In [ ]:
!head -9 nisqually_2001.csv

<div class="alert alert-info">

### Exercise 3: Load and explore

1. Load each file into a 2D array (`nis` and `nor`) and print their shapes. How many events are in each?
2. Pull out the columns you'll need as their own 1D arrays: hours, depth, magnitude, and is_earthquake for **both** earthquakes. Use names like `nis_hours` and `nor_mag` so you never mix them up.
3. How many events in each file are **explosions**?
4. Some Northridge magnitudes are missing. Count them with `np.isnan()`, which returns a mask that is `True` wherever an array holds `nan`.
5. Predict, then run: `print(np.nan == np.nan)`. Why can't you count missing values with `nor_mag == np.nan`?

</div>

In [ ]:
# your answer here
# 1
nis = np.loadtxt("nisqually_2001.csv", delimiter=",")
nor = ___
print(nis.shape, nor.shape)

# 2: columns are 0 hours, 3 depth, 4 magnitude, 5 is_earthquake
nis_hours = nis[:, 0]
nis_depth = nis[:, ___]
nis_mag = ___
nis_eq = ___

nor_hours = ___
nor_depth = ___
nor_mag = ___
nor_eq = ___

# 3
print("Nisqually explosions: ", np.sum(nis_eq == ___))
print("Northridge explosions:", ___)

# 4
print("Missing Northridge magnitudes:", np.sum(___))

# 5


<div class="alert alert-info">

### Exercise 4: Which events are aftershocks?

For this lab, an **aftershock** is an event that is (a) an earthquake, (b) after the mainshock, and (c) within the first 7 days (168 hours).

1. Make two masks, `nis_after` and `nor_after`, combining all three conditions. How many aftershocks does each earthquake have? (Look at `nor_hours[:5]` first. Why do we need condition (b)?)
2. For each earthquake, count the aftershocks of magnitude **2 or more**, **3 or more**, and **4 or more**.
3. Find the **largest** Northridge aftershock. First try `nor_mag[nor_after].max()`, and read what you get. Then use `np.nanmax()` instead, the same idea as `np.nanmean()` from Friday.
4. **When** was it, and how **deep**? Use `np.nanargmax()` on the aftershock magnitudes, and use that position on the aftershock **hours** and **depths**. Report the time in **minutes** after the mainshock.

</div>

In [ ]:
# your answer here
# 1
print(nor_hours[:5].round(2))

nis_after = (nis_eq == 1) & (nis_hours ___ 0) & (nis_hours ___ 168)
nor_after = ___
print("Nisqually aftershocks: ", np.sum(nis_after))
print("Northridge aftershocks:", ___)

# 2
for m in [2, 3, 4]:
    nis_count = np.sum(nis_mag[nis_after] >= m)
    nor_count = ___
    print(f"M{m}+:  Nisqually {nis_count}   Northridge {nor_count}")

# 3
print(nor_mag[nor_after].max())
print(___)

# 4
after_mag = nor_mag[nor_after]
after_hours = nor_hours[nor_after]
after_depth = ___
i = ___
print(f"M{after_mag[i]}, {___:.1f} minutes after, {___:.1f} km deep")


<div class="alert alert-warning">

**Question 3:** Compare the two aftershock counts. Northridge had about 1,500 aftershocks of M2 or more in its first week. How many did Nisqually have?

</div>

> your answer here

<div class="alert alert-info">

### Exercise 5: How deep were the aftershocks?

1. Print the **depths** and **magnitudes** of all of Nisqually's aftershocks.
2. Nisqually's mainshock was 51.8 km deep. Make a mask for aftershocks **deeper than 40 km**. How many are there, and what are their magnitudes?
3. Northridge's mainshock was 18.2 km deep. What was the **average** aftershock depth? What **fraction** of its aftershocks were shallower than the mainshock? (Hint: the `.mean()` of a boolean array is the fraction that are `True`, since `True` counts as 1.)

</div>

In [ ]:
# your answer here


<div class="alert alert-warning">

**Question 4:** Six of Nisqually's "aftershocks" were 8–28 km deep, 25–45 km **above** the mainshock. Do you think they were really aftershocks? What else could they be, and what extra information would help you decide?

</div>

> your answer here

<div class="alert alert-info">

### Exercise 6: The energy budget

1. Use `quake_energy()` on **all** of each earthquake's aftershock magnitudes at once, and add them up. (Missing magnitudes give missing energies, so use `np.nansum()`.)
2. What **percent** of the total energy (mainshock + first-week aftershocks) came from the mainshock alone? Print it with 3 decimal places.

</div>

In [ ]:
# your answer here


<div class="alert alert-info">

### Exercise 7: Day by day

Use a `for` loop over the 7 days with `range()`, and a mask inside the loop, to count Northridge's aftershocks on each day: day 1 is hours 0–24, day 2 is hours 24–48, and so on. Print one line per day. **Write your plan as comments first.**

</div>

In [ ]:
# plan (write it in comments first):
#
#
# your answer here


<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab06_LastName.ipynb
git commit -m "Finish Exercises 3-7"
git push
```

</div>

---
## Part 4: Waves

A seismometer records ground motion as a **wave**. The simplest wave is a sine wave:

$$y = A \sin(2\pi f t)$$

where $A$ is the **amplitude** (how big the wave is), $f$ is the **frequency** in hertz (cycles per second), and $t$ is time in seconds. The **period**, $T = 1/f$, is the number of seconds one cycle takes.

The code cell under Exercise 8 already has the code to plot one. You've seen `plt.plot()` with `label=` and `plt.legend()` before; we'll learn plotting properly soon.

<div class="alert alert-info">

### Exercise 8: One wave at a time

1. Run the code that's already in the cell. How many full cycles fit in 2 seconds? What is the period?
2. Add a second wave to the **same** plot with $A = 0.5$ and $f = 3$ Hz, with its own label.
3. What is the time step between samples in `t`? (Hint: `t[1] - t[0]`.) How many samples per second is that?

</div>

In [ ]:
# your answer here
# 1: run this first
t = np.linspace(0, 2, 201)          # 2 seconds of time
y = np.sin(2 * np.pi * 1 * t)       # A = 1, f = 1 Hz

plt.plot(t, y, label="1 Hz")
plt.xlabel("time (s)")
plt.ylabel("ground motion")
plt.legend()
plt.show()

# 2


# 3


<div class="alert alert-info">

### Exercise 9: A bank of waves, with broadcasting

1. Make `freqs = np.array([0.5, 1, 2, 4])`. **Predict** the shape of `freqs[:, np.newaxis] * t`, then use it to make `waves`, one sine wave per **row**, all with amplitude 1.
2. Use a `for` loop over `zip(freqs, waves)` to plot all four on one plot, each with a label like `"2.0 Hz"`.
3. Real ground motion is many waves **added together**. Make `total`, the sum of all four waves at each time. (Which `axis`?) Plot it.
4. What is the biggest value of `total`, and at what time does it happen?

</div>

In [ ]:
# your answer here


<div class="alert alert-info">

### Exercise 10: Sampling too slowly

A seismometer can only record the ground at fixed moments. What happens if it doesn't sample often enough?

1. Make `t_coarse = np.linspace(0, 2, 11)`. How many samples per second is that?
2. On one plot, draw the 4 Hz wave on the fine `t` **and** on `t_coarse`. For the coarse one, use `plt.plot(t_coarse, y_coarse, "o-", label="5 samples/s")` so you can see the individual samples.
3. How many cycles does the coarse version **appear** to show in 2 seconds? What frequency does it look like?

</div>

In [ ]:
# your answer here


<div class="alert alert-warning">

**Question 5:** Look back at the whole lab. In three or four sentences, explain why Nisqually and Northridge had such different consequences, using at least two numbers you calculated.

</div>

> your answer here

# End for Undergraduate Students
Graduate students should continue to the exercise below.

## Submitting
Save your notebook (ctrl + S), then in the terminal from your Week 07 repo folder:
```
git status
git add lab06_LastName.ipynb
git commit -m "[insert a message here]"
git push
```
Check your repo's page on GitHub to make sure your commit arrived. You can push as many times as you like before the deadline; your most recent push is what counts.

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

### For Graduate Students Only: real seismograms
Sine waves are tidy. Real ground motion isn't. To see a real seismogram you'll use **ObsPy**, a Python package for seismology. It's one of many science packages built on top of NumPy, so you'll meet this pattern again: someone else wrote the package, it hands you NumPy arrays, and you use the tools you already know. Don't get bogged down in how ObsPy works inside; we'll only use the three things below.

First install it. You only need to do this once, and it takes about 30 seconds. A lot of text will scroll by, which is normal. Run this:

In [ ]:
%pip install obspy

Then load ObsPy's built in example, a small earthquake recorded in Germany in 2009. (It isn't Nisqually or Northridge, but it's a real seismogram.)

In [ ]:
from obspy import read

st = read()  # three recordings of the same earthquake
tr = st[0]   # the first one: the vertical (up-down) ground motion
tr.plot()
print(tr.stats)    #sampling_rate, npts, station, etc.

Now let's download the real seismograms of the two earthquakes from a public seismology archive. Each record starts 30 seconds **before** the earthquake began and runs for 270 seconds after it.

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")     # the public seismology archive (now EarthScope)

# Nisqually: station GNW (Federal Way, WA), vertical component
t_nis = UTCDateTime("2001-02-28T18:54:32")
st_nis = client.get_waveforms("UW", "GNW", "*", "BHZ", t_nis - 30, t_nis + 270)

# Northridge: station PAS (Pasadena, CA), vertical component
t_nor = UTCDateTime("1994-01-17T12:30:55")
st_nor = client.get_waveforms("CI", "PAS", "*", "BHZ", t_nor - 30, t_nor + 270)

Again, all you need to know:
- `tr.data` is a NumPy array of ground motion. It's in raw instrument "counts", so there are no units.
- `tr.times` is a NumPy array of the time of each sample in seconds.
- `tr.stats.sampling_rate` is how many samples per second the instrument recorded.

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

## Exercise G1 : The Nisqually and Northridge seismograms

Let `tr_nis = st_nis[0]` and `tr_nor = st_nor[0]`. Do steps 1 to 4 for **both** earthquakes. A `for` loop over `[("Nisqually", tr_nis), ("Northridge", tr_nor)]` saves typing.

1. Print the `type`, `shape`, and `dtype` of `.data`, and print `.stats.sampling_rate`. Then find the time step and samples per second from `.times()` the same way you did in Exercise 8, and check that they agree.
2. Plot the data against `.times()`. Roughly when does the strong shaking start and stop? (Remember that the earthquake began at **30 s** in each record.)
3. What is the largest ground motion, and when does it happen? (Careful: a big *negative* swing is just as strong as a big positive one. Hint: `np.abs`.) How many seconds **after the earthquake began** did it arrive?
4. Use a boolean mask to find the samples where the motion is more than **half** of the largest value. For how many seconds is the ground moving that hard? (Each sample is one time step long.) What fraction of the whole record is that?
5. Compare your two earthquakes. Which had the larger peak in counts, and why isn't that a fair comparison of how hard each one shook? Which one had longer strong shaking, and by how many seconds?


</div>

In [ ]:
tr_nis, tr_nor = st_nis[0], st_nor[0]

# your answer here


<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">


## Exercise G2 : Three components and sampling again

Now switch to ObsPy's built-in example, a small earthquake recorded in Germany in 2009. The code below loads it. It has three recordings of the same event, `st[0]`, `st[1]`, and `st[2]`: up-down (`EHZ`), north-south (`EHN`), and east-west (`EHE`).
```python
from obspy import read

st = read()
tr = st[0]
t_real = tr.times()
```

1. Stack the three `.data` arrays into one 2D array, `data3`, with one component per **row**. (Hint: `np.array([st[0].data, st[1].data, st[2].data])`.) **Predict** its shape first.
2. Which component moved the most? Find the largest absolute value on **each** row at once. (Which `axis`?)
3. The total ground motion at each moment combines all three directions: square each value, add the three components, and take the square root. Make `total_motion`, plot it against `t_real`, and print its biggest value and when it happens.
4. Now revisit Exercise 10. Keep only **every 20th** sample with slicing, `tr.data[::20]` and `t_real[::20]`. That's 5 samples per second, as in Exercise 10. Plot just the window from 4 to 8 s, with the full data as a line and the coarse data as `"o-"`.


</div>

In [ ]:
# setup: ObsPy's built-in example earthquake (three recordings of one event)
from obspy import read

st = read()
tr = st[0]
t_real = tr.times()

# your answer here


# End for Graduate Students

## Submitting
Save your notebook (ctrl + S), then in the terminal from your Week 07 repo folder:
```
git status
git add lab06_LastName.ipynb
git commit -m "[insert a message here]"
git push
```
Check your repo's page on GitHub to make sure your commit arrived. You can push as many times as you like before the deadline; your most recent push is what counts.

---
## References

- Earthquake Engineering Research Institute (2001). *Preliminary Observations on the Nisqually Earthquake of February 28, 2001.* [PDF](https://www.eeri.org/lfe/pdf/usa_nisqually_preliminary_report.pdf)
- Gutenberg, B., & Richter, C. F. (1944). Frequency of earthquakes in California. *Bulletin of the Seismological Society of America*, 34(4), 185–188.
- Gutenberg, B., & Richter, C. F. (1956). Magnitude and energy of earthquakes. *Annali di Geofisica*, 9, 1–15.
- Okal, E. A. (2019). Energy and magnitude: A historical perspective. *Pure and Applied Geophysics*, 176, 3815–3849.
- Pacific Northwest Seismograph Network (2001). Quarterly network report. [Link](https://assets.pnsn.org/legacy_reports/Sum01/q01A.html)
- U.S. Geological Survey (1996). *USGS Response to an Urban Earthquake — Northridge '94*, Open-File Report 96-263. [Link](https://pubs.usgs.gov/of/1996/ofr-96-0263/introduc.htm)
- U.S. Geological Survey, ANSS Comprehensive Earthquake Catalog (ComCat). [Link](https://earthquake.usgs.gov/data/comcat/)